In [ ]:
import pandas as pd

print("Library pandas imported.")

In [ ]:
columns_to_discard = [
    'collision_index',
    'collision_ref_no',
    'location_easting_osgr',
    'location_northing_osgr',
    'local_authority_ons_district',
    'local_authority_highway',
    'local_authority_highway_current',
    'lsoa_of_accident_location',
    'AREFNO'
]

data_frames = []

# Find all 'year.txt' files in the current directory
file_paths = glob.glob('*.csv')

if not file_paths:
    print("No 'year.txt' files found in the current directory. Please make sure the data files are present.")
else:
    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            # Drop specified columns, ensuring they exist in the DataFrame
            cols_present = [col for col in columns_to_discard if col in df.columns]
            if cols_present:
                df = df.drop(columns=cols_present, axis=1)
            else:
                print(f"Warning: No specified columns to discard found in {file_path}.")
            data_frames.append(df)
            print(f"Processed {file_path} and added to list. Shape: {df.shape}")
        except Exception as e:
            print(f"Error processing {file_path}: {e}")

print(f"Total processed DataFrames: {len(data_frames)}")

In [ ]:
merged_df = pd.concat(data_frames, ignore_index=True)
print(f"Merged DataFrame shape: {merged_df.shape}")


In [ ]:
merged_df.to_csv('merged_collision_data.csv', index=False)
print("Merged DataFrame saved to 'merged_collision_data.csv'")

In [ ]:
merged_df = pd.read_csv('merged_collision_data.csv', low_memory=False)
print("First 5 rows of merged_df (reloaded for target change):")
print(merged_df.head())
print(f"Shape of merged_df (reloaded): {merged_df.shape}")

In [ ]:
print("DataFrame Info:")
merged_df.info()

print("\nMissing values per column:")
print(merged_df.isnull().sum()[merged_df.isnull().sum() > 0])

print("\nValue counts for 'collision_severity':")
print(merged_df['collision_severity'].value_counts())

**Reasoning**:
The next step is to handle missing values in the target variable 'collision_severity' by dropping rows where it is NaN, and then converting the column to an integer type, as per the instructions. This ensures the target variable is clean and in the correct format for modeling.



In [ ]:
# Drop rows where 'Severity' is NaN, as it is the new target variable
merged_df.dropna(subset=['Severity'], inplace=True)

# Normalize 'Severity' column by converting to uppercase
merged_df['Severity'] = merged_df['Severity'].str.upper()

# Drop the old target variable 'collision_severity' as it is no longer the target
if 'collision_severity' in merged_df.columns:
    merged_df.drop(columns=['collision_severity'], inplace=True)

print("Missing values in 'Severity' after dropping NaNs:")
print(merged_df['Severity'].isnull().sum())
print("Data type of 'Severity' after handling (should be object before Label Encoding):")
print(merged_df['Severity'].dtype)
print("Value counts for 'Severity' after cleaning and normalization:")
print(merged_df['Severity'].value_counts())

In [ ]:
if 'date' in merged_df.columns and 'time' in merged_df.columns:
    merged_df['date'] = pd.to_datetime(merged_df['date'], format='%d/%m/%Y', errors='coerce')
    merged_df['time'] = pd.to_datetime(merged_df['time'], format='%H:%M:%S', errors='coerce')

    merged_df['hour_of_day'] = merged_df['time'].dt.hour
    merged_df['month'] = merged_df['date'].dt.month

    merged_df.drop(columns=['date', 'time'], inplace=True)

    print("New features 'hour_of_day' and 'month' created.")
    print("Original 'date' and 'time' columns dropped.")
    print("First 5 rows with new features:")
    print(merged_df[['hour_of_day', 'month']].head())
else:
    print("'date' or 'time' columns not found in DataFrame. Assuming they have already been processed or dropped.")

In [ ]:
print("Number of columns before dropping all-NaN columns:", merged_df.shape[1])

# Identify columns with 0 non-null values
columns_to_drop_all_nan = merged_df.columns[merged_df.isnull().all()].tolist()

if columns_to_drop_all_nan:
    merged_df.drop(columns=columns_to_drop_all_nan, inplace=True)
    print(f"Dropped {len(columns_to_drop_all_nan)} columns that were entirely NaN: {columns_to_drop_all_nan}")
else:
    print("No columns were entirely NaN to drop.")

print("Number of columns after dropping all-NaN columns:", merged_df.shape[1])

# Re-identify numerical and categorical columns after dropping all-NaN columns
# Exclude 'Severity' from categorical features as it is now the target variable
numerical_cols = merged_df.select_dtypes(include=['number']).columns.tolist()
categorical_cols = merged_df.select_dtypes(exclude=['number']).columns.tolist()

if 'Severity' in categorical_cols:
    categorical_cols.remove('Severity') # Ensure target is not processed as a feature

print(f"Number of numerical columns (after dropping all-NaN and excluding target): {len(numerical_cols)}")
print(f"Number of categorical columns (after dropping all-NaN and excluding target): {len(categorical_cols)}")

# Re-impute remaining missing values for numerical columns with the mean
for col in numerical_cols:
    if merged_df[col].isnull().any():
        merged_df[col] = merged_df[col].fillna(merged_df[col].mean())

# Re-impute remaining missing values for categorical columns with the mode or 'Unknown'
for col in categorical_cols:
    if merged_df[col].isnull().any():
        mode_val = merged_df[col].mode()
        if not mode_val.empty:
            merged_df[col] = merged_df[col].fillna(mode_val[0])
        else:
            merged_df[col] = merged_df[col].fillna('Unknown')

print("Missing values after re-imputation (expected to be 0 for features):")
print(merged_df.drop(columns=['Severity']).isnull().sum().sum()) # Check features only

# Apply one-hot encoding to all identified categorical features (excluding 'Severity')
merged_df = pd.get_dummies(merged_df, columns=categorical_cols, drop_first=True)

print("Categorical features one-hot encoded.")
print(f"Shape of DataFrame after one-hot encoding: {merged_df.shape}")

In [ ]:
from sklearn.preprocessing import StandardScaler

# Re-identify numerical columns after one-hot encoding, excluding the target variable 'Severity'
numerical_cols = merged_df.select_dtypes(include=['number']).columns.tolist()
if 'Severity' in numerical_cols:
    numerical_cols.remove('Severity') # Ensure target is not scaled

# Initialize StandardScaler
scaler = StandardScaler()

# Apply scaling to numerical features
merged_df[numerical_cols] = scaler.fit_transform(merged_df[numerical_cols])

print("Numerical features scaled using StandardScaler.")
print("First 5 rows of scaled numerical features (excluding Severity):")
print(merged_df[numerical_cols].head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Split the data into features (X) and target (y)
X = merged_df.drop('Severity', axis=1) # Drop the new target from features
y = merged_df['Severity'] # New target variable

print(f"Shape of features (X): {X.shape}")
print(f"Shape of target (y): {y.shape}")

# Visualize the distribution of the target variable y
plt.figure(figsize=(8, 6))
sns.countplot(x=y, palette='viridis')
plt.title('Distribution of Collision Severity (Severity)') # Updated title
plt.xlabel('Collision Severity (Severity)') # Updated label
plt.ylabel('Count')
plt.show()

print("Data split into features (X) and target (y), and target distribution visualized.")

In [ ]:
if 'date' in merged_df.columns and 'time' in merged_df.columns:
    merged_df['date'] = pd.to_datetime(merged_df['date'], format='%d/%m/%Y', errors='coerce')
    merged_df['time'] = pd.to_datetime(merged_df['time'], format='%H:%M:%S', errors='coerce')

    merged_df['hour_of_day'] = merged_df['time'].dt.hour
    merged_df['month'] = merged_df['date'].dt.month

    merged_df.drop(columns=['date', 'time'], inplace=True)

    print("New features 'hour_of_day' and 'month' created.")
    print("Original 'date' and 'time' columns dropped.")
    print("First 5 rows with new features:")
    print(merged_df[['hour_of_day', 'month']].head())
else:
    print("'date' or 'time' columns not found in DataFrame. Assuming they have already been processed or dropped.")

In [ ]:
print("Number of columns before dropping all-NaN columns:", merged_df.shape[1])

# Identify columns with 0 non-null values
columns_to_drop_all_nan = merged_df.columns[merged_df.isnull().all()].tolist()

if columns_to_drop_all_nan:
    merged_df.drop(columns=columns_to_drop_all_nan, inplace=True)
    print(f"Dropped {len(columns_to_drop_all_nan)} columns that were entirely NaN: {columns_to_drop_all_nan}")
else:
    print("No columns were entirely NaN to drop.")

print("Number of columns after dropping all-NaN columns手間:", merged_df.shape[1])

# Re-identify numerical and categorical columns after dropping all-NaN columns
# Exclude 'Severity' from categorical features as it is now the target variable
numerical_cols = merged_df.select_dtypes(include=['number']).columns.tolist()
categorical_cols = merged_df.select_dtypes(exclude=['number']).columns.tolist()

if 'Severity' in categorical_cols:
    categorical_cols.remove('Severity') # Ensure target is not processed as a feature

print(f"Number of numerical columns (after dropping all-NaN and excluding target): {len(numerical_cols)}")
print(f"Number of categorical columns (after dropping all-NaN and excluding target): {len(categorical_cols)}")

# Re-impute remaining missing values for numerical columns with the mean
for col in numerical_cols:
    if merged_df[col].isnull().any():
        merged_df[col] = merged_df[col].fillna(merged_df[col].mean())

# Re-impute remaining missing values for categorical columns with the mode or 'Unknown'
for col in categorical_cols:
    if merged_df[col].isnull().any():
        mode_val = merged_df[col].mode()
        if not mode_val.empty:
            merged_df[col] = merged_df[col].fillna(mode_val[0])
        else:
            merged_df[col] = merged_df[col].fillna('Unknown')

print("Missing values after re-imputation (expected to be 0 for features):")
print(merged_df.drop(columns=['Severity']).isnull().sum().sum()) # Check features only

# Apply one-hot encoding to all identified categorical features (excluding 'Severity')
merged_df = pd.get_dummies(merged_df, columns=categorical_cols, drop_first=True)

print("Categorical features one-hot encoded.")
print(f"Shape of DataFrame after one-hot encoding: {merged_df.shape}")

In [ ]:
from sklearn.preprocessing import StandardScaler

# Re-identify numerical columns after one-hot encoding, excluding the target variable 'Severity'
numerical_cols = merged_df.select_dtypes(include=['number']).columns.tolist()
if 'Severity' in numerical_cols:
    numerical_cols.remove('Severity') # Ensure target is not scaled

# Initialize StandardScaler
scaler = StandardScaler()

# Apply scaling to numerical features
merged_df[numerical_cols] = scaler.fit_transform(merged_df[numerical_cols])

print("Numerical features scaled using StandardScaler.")
print("First 5 rows of scaled numerical features (excluding Severity):")
print(merged_df[numerical_cols].head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Split the data into features (X) and target (y)
X = merged_df.drop('Severity', axis=1) # Drop the new target from features
y = merged_df['Severity'] # New target variable

print(f"Shape of features (X): {X.shape}")
print(f"Shape of target (y): {y.shape}")

# Visualize the distribution of the target variable y
plt.figure(figsize=(8, 6))
sns.countplot(x=y, palette='viridis')
plt.title('Distribution of Collision Severity (Severity)') # Updated title
plt.xlabel('Collision Severity (Severity)') # Updated label
plt.ylabel('Count')
plt.show()

print("Data split into features (X) and target (y), and target distribution visualized.")

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder

# Convert target variable to NumPy array for Keras compatibility
y_np = y.values

# Encode target labels to 0-indexed values for sparse_categorical_crossentropy
# 'Severity' has string values (e.g., 'Slight', 'Serious'). We need to map them to 0, 1, 2.
le = LabelEncoder()
y_encoded = le.fit_transform(y_np)

print("Necessary libraries imported and target variable converted to NumPy array and 0-indexed.")
print(f"Original target unique values: {np.unique(y_np)}")
print(f"Encoded target unique values: {np.unique(y_encoded)}")

In [ ]:
def create_model(input_dim, num_classes):
    model = keras.Sequential([
        keras.layers.Dense(128, activation='relu', input_shape=(input_dim,)),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Determine input dimension and number of classes
input_dim = X.shape[1] # X is now correctly updated after preprocessing
num_classes = len(np.unique(y_encoded)) # Based on newly encoded y

print(f"Input dimension for the model: {input_dim}")
print(f"Number of output classes: {num_classes}")

# Create a sample model to print its summary
sample_model = create_model(input_dim, num_classes)
print("Neural Network Model Summary:")
sample_model.summary()
print("Neural network model architecture defined and summary displayed.")

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

kf = KFold(n_splits=5, shuffle=True, random_state=42)

all_accuracies = []
all_losses = []

# Ensure all columns in X are numeric and handle any remaining non-numeric values
for col in X.columns:
    # Attempt to convert column to numeric, coercing any unconvertible values to NaN
    X[col] = pd.to_numeric(X[col], errors='coerce')

# After coercing, impute any newly introduced NaNs (from conversion failures)
# with the mean of their respective columns.
# We ensure to only calculate mean for numeric columns.
X.fillna(X.mean(numeric_only=True), inplace=True)

# Convert X to numpy array for Keras compatibility, explicitly setting dtype
X_np = X.values.astype(np.float32)

print("X has been converted to a purely numerical numpy array (float32) for Keras compatibility.")
print(f"Shape of X_np: {X_np.shape}, Dtype: {X_np.dtype}")

print("Starting 5-fold cross-validation...")

for fold, (train_index, val_index) in enumerate(kf.split(X_np, y_encoded)):
    print(f"\n--- Fold {fold+1}/5 ---")

    # X_train and X_val are already slices of X_np (which is float32), so no further .astype() is needed here.
    X_train, X_val = X_np[train_index], X_np[val_index]
    y_train, y_val = y_encoded[train_index], y_encoded[val_index]

    model = create_model(input_dim, num_classes)

    # Train the model
    history = model.fit(
        X_train,
        y_train,
        epochs=10,  # You can adjust the number of epochs
        batch_size=32, # You can adjust the batch size
        validation_data=(X_val, y_val),
        verbose=0 # Set to 1 or 2 for more detailed output during training
    )

    # Evaluate the model on the validation set
    loss, accuracy = model.evaluate(X_val, y_val, verbose=0)
    print(f"Fold {fold+1} - Validation Loss: {loss:.4f}, Validation Accuracy: {accuracy:.4f}")

    all_losses.append(loss)
    all_accuracies.append(accuracy)

print("\n--- Cross-validation complete ---")
print(f"Average Validation Accuracy: {np.mean(all_accuracies):.4f} (+/- {np.std(all_accuracies):.4f})")
print(f"Average Validation Loss: {np.mean(all_losses):.4f} (+/- {np.std(all_losses):.4f})")

# Plotting training history for the last fold (as an example)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy - Last Fold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss - Last Fold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

print("Cross-validation results and last fold's training history plotted.")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Get predictions for the last validation fold
y_pred_prob = model.predict(X_val)
y_pred = np.argmax(y_pred_prob, axis=1)

print("\n--- Detailed Evaluation for the Last Fold ---")

# Dynamically get unique labels and their names for the current fold
unique_labels_in_fold = np.unique(y_val)
target_names_in_fold = le.inverse_transform(unique_labels_in_fold)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_val, y_pred, labels=unique_labels_in_fold, target_names=target_names_in_fold))

# Confusion Matrix
conf_matrix = confusion_matrix(y_val, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names_in_fold, yticklabels=target_names_in_fold) # Use dynamically derived names
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Last Fold (Severity)') # Updated title
plt.show()

print("Detailed classification report and confusion matrix displayed for the last validation fold.")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Get predictions for the last validation fold
y_pred_prob = model.predict(X_val)
y_pred = np.argmax(y_pred_prob, axis=1)

print("\n--- Detailed Evaluation for the Last Fold ---")

# Dynamically get unique labels and their names for the current fold
unique_labels_in_fold = np.unique(y_val)
target_names_in_fold = le.inverse_transform(unique_labels_in_fold)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_val, y_pred, labels=unique_labels_in_fold, target_names=target_names_in_fold))

# Confusion Matrix
conf_matrix = confusion_matrix(y_val, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names_in_fold, yticklabels=target_names_in_fold) # Use dynamically derived names
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Last Fold (Severity)') # Updated title
plt.show()

print("Detailed classification report and confusion matrix displayed for the last validation fold.")

## Final Task

### Subtask:
Provide a summary of the neural network model's performance based on the 5-fold cross-validation results.

## Summary:

### Q&A
The neural network model achieved an average validation accuracy of 0.9999 (\+/- 0.0001) and an average validation loss of 0.0003 (\+/- 0.0004) across 5-fold cross-validation. For the last validation fold, the model demonstrated perfect precision, recall, and F1-score for all three `Severity` classes ('Fatal', 'Serious', 'Slight'), indicating flawless classification on that specific fold.

### Data Analysis Key Findings
*   The initial dataset (`merged_collision_data.csv`) contained 29,835 rows and 99 columns. After reloading, rows with missing 'Severity' were removed, reducing the dataset to 9,679 entries. The target variable 'Severity' has string values (e.g., 'Slight', 'Serious', 'Fatal') and was encoded into 0-indexed numerical values.
*   The distribution of the 'Severity' target variable was:
    *   SLIGHT: 4396
    *   Slight: 3826
    *   SERIOUS: 963
    *   Serious: 473
    *   Fatal: 15
    *   FATAL: 6
*   Feature engineering successfully extracted 'hour_of_day' and 'month' from 'date' and 'time' columns, which were then dropped.
*   A total of 51 columns that were entirely `NaN` were dropped. Remaining numerical missing values were imputed with the mean, and categorical missing values with the mode or 'Unknown'.
*   Categorical features were one-hot encoded, leading to a DataFrame with 10,801 columns. Numerical features were scaled using `StandardScaler`.
*   The neural network showed exceptionally high performance with an average validation accuracy of 0.9999, even after addressing issues with mixed data types and dynamic class label handling.

### Insights or Next Steps
*   The exceptionally high model performance (near-perfect accuracy and F1-scores across all classes) still warrants further investigation to rule out potential issues like data leakage or overfitting, which can occur with such high performance in complex, real-world data.
*   Consider performing data cleaning and standardization of the 'Severity' categories (e.g., merging 'SLIGHT' and 'Slight', 'SERIOUS' and 'Serious', 'FATAL' and 'Fatal') to reduce the number of target classes, which might simplify the classification problem and provide a clearer interpretation of the model's performance.
*   Evaluate the model on an entirely unseen test set to ensure its generalization capabilities are robust and not solely optimized for the cross-validation folds.


**Reasoning**:
The previous code executed successfully, completing the data preprocessing and visualization of the target variable. The next logical step, as per the overall task, is to define and train a neural network model using 5-fold cross-validation. This requires importing libraries for neural network construction and cross-validation, and converting the target variable `y` to a NumPy array for compatibility with Keras/TensorFlow.



In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder

# Convert target variable to NumPy array for Keras compatibility
y_np = y.values

# Encode target labels to 0-indexed values for sparse_categorical_crossentropy
# 'Severity' has string values (e.g., 'Slight', 'Serious'). We need to map them to 0, 1, 2.
le = LabelEncoder()
y_encoded = le.fit_transform(y_np)

print("Necessary libraries imported and target variable converted to NumPy array and 0-indexed.")
print(f"Original target unique values: {np.unique(y_np)}")
print(f"Encoded target unique values: {np.unique(y_encoded)}")

In [ ]:
def create_model(input_dim, num_classes):
    model = keras.Sequential([
        keras.layers.Dense(128, activation='relu', input_shape=(input_dim,)),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Determine input dimension and number of classes
input_dim = X.shape[1] # X is now correctly updated after preprocessing
num_classes = len(np.unique(y_encoded)) # Based on newly encoded y

print(f"Input dimension for the model: {input_dim}")
print(f"Number of output classes: {num_classes}")

# Create a sample model to print its summary
sample_model = create_model(input_dim, num_classes)
print("Neural Network Model Summary:")
sample_model.summary()
print("Neural network model architecture defined and summary displayed.")

In [ ]:
if 'date' in merged_df.columns and 'time' in merged_df.columns:
    merged_df['date'] = pd.to_datetime(merged_df['date'], format='%d/%m/%Y', errors='coerce')
    merged_df['time'] = pd.to_datetime(merged_df['time'], format='%H:%M:%S', errors='coerce')

    merged_df['hour_of_day'] = merged_df['time'].dt.hour
    merged_df['month'] = merged_df['date'].dt.month

    merged_df.drop(columns=['date', 'time'], inplace=True)

    print("New features 'hour_of_day' and 'month' created.")
    print("Original 'date' and 'time' columns dropped.")
    print("First 5 rows with new features:")
    print(merged_df[['hour_of_day', 'month']].head())
else:
    print("'date' or 'time' columns not found in DataFrame. Assuming they have already been processed or dropped.")

In [ ]:
print("Number of columns before dropping all-NaN columns:", merged_df.shape[1])

# Identify columns with 0 non-null values
columns_to_drop_all_nan = merged_df.columns[merged_df.isnull().all()].tolist()

if columns_to_drop_all_nan:
    merged_df.drop(columns=columns_to_drop_all_nan, inplace=True)
    print(f"Dropped {len(columns_to_drop_all_nan)} columns that were entirely NaN: {columns_to_drop_all_nan}")
else:
    print("No columns were entirely NaN to drop.")

print("Number of columns after dropping all-NaN columns手間:", merged_df.shape[1])

# Re-identify numerical and categorical columns after dropping all-NaN columns
# Exclude 'Severity' from categorical features as it is now the target variable
numerical_cols = merged_df.select_dtypes(include=['number']).columns.tolist()
categorical_cols = merged_df.select_dtypes(exclude=['number']).columns.tolist()

if 'Severity' in categorical_cols:
    categorical_cols.remove('Severity') # Ensure target is not processed as a feature

print(f"Number of numerical columns (after dropping all-NaN and excluding target): {len(numerical_cols)}")
print(f"Number of categorical columns (after dropping all-NaN and excluding target): {len(categorical_cols)}")

# Re-impute remaining missing values for numerical columns with the mean
for col in numerical_cols:
    if merged_df[col].isnull().any():
        merged_df[col] = merged_df[col].fillna(merged_df[col].mean())

# Re-impute remaining missing values for categorical columns with the mode or 'Unknown'
for col in categorical_cols:
    if merged_df[col].isnull().any():
        mode_val = merged_df[col].mode()
        if not mode_val.empty:
            merged_df[col] = merged_df[col].fillna(mode_val[0])
        else:
            merged_df[col] = merged_df[col].fillna('Unknown')

print("Missing values after re-imputation (expected to be 0 for features):")
print(merged_df.drop(columns=['Severity']).isnull().sum().sum()) # Check features only

# Apply one-hot encoding to all identified categorical features (excluding 'Severity')
merged_df = pd.get_dummies(merged_df, columns=categorical_cols, drop_first=True)

print("Categorical features one-hot encoded.")
print(f"Shape of DataFrame after one-hot encoding: {merged_df.shape}")

In [ ]:
from sklearn.preprocessing import StandardScaler

# Re-identify numerical columns after one-hot encoding, excluding the target variable 'Severity'
numerical_cols = merged_df.select_dtypes(include=['number']).columns.tolist()
if 'Severity' in numerical_cols:
    numerical_cols.remove('Severity') # Ensure target is not scaled

# Initialize StandardScaler
scaler = StandardScaler()

# Apply scaling to numerical features
merged_df[numerical_cols] = scaler.fit_transform(merged_df[numerical_cols])

print("Numerical features scaled using StandardScaler.")
print("First 5 rows of scaled numerical features (excluding Severity):")
print(merged_df[numerical_cols].head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Split the data into features (X) and target (y)
X = merged_df.drop('Severity', axis=1) # Drop the new target from features
y = merged_df['Severity'] # New target variable

print(f"Shape of features (X): {X.shape}")
print(f"Shape of target (y): {y.shape}")

# Visualize the distribution of the target variable y
plt.figure(figsize=(8, 6))
sns.countplot(x=y, palette='viridis')
plt.title('Distribution of Collision Severity (Severity)') # Updated title
plt.xlabel('Collision Severity (Severity)') # Updated label
plt.ylabel('Count')
plt.show()

print("Data split into features (X) and target (y), and target distribution visualized.")

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder

# Convert target variable to NumPy array for Keras compatibility
y_np = y.values

# Encode target labels to 0-indexed values for sparse_categorical_crossentropy
# 'Severity' has string values (e.g., 'Slight', 'Serious'). We need to map them to 0, 1, 2.
le = LabelEncoder()
y_encoded = le.fit_transform(y_np)

print("Necessary libraries imported and target variable converted to NumPy array and 0-indexed.")
print(f"Original target unique values: {np.unique(y_np)}")
print(f"Encoded target unique values: {np.unique(y_encoded)}")

In [ ]:
def create_model(input_dim, num_classes):
    model = keras.Sequential([
        keras.layers.Dense(128, activation='relu', input_shape=(input_dim,)),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Determine input dimension and number of classes
input_dim = X.shape[1] # X is now correctly updated after preprocessing
num_classes = len(np.unique(y_encoded)) # Based on newly encoded y

print(f"Input dimension for the model: {input_dim}")
print(f"Number of output classes: {num_classes}")

# Create a sample model to print its summary
sample_model = create_model(input_dim, num_classes)
print("Neural Network Model Summary:")
sample_model.summary()
print("Neural network model architecture defined and summary displayed.")

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

kf = KFold(n_splits=5, shuffle=True, random_state=42)

all_accuracies = []
all_losses = []

# Ensure all columns in X are numeric and handle any remaining non-numeric values
for col in X.columns:
    # Attempt to convert column to numeric, coercing any unconvertible values to NaN
    X[col] = pd.to_numeric(X[col], errors='coerce')

# After coercing, impute any newly introduced NaNs (from conversion failures)k
# with the mean of their respective columns.
# We ensure to only calculate mean for numeric columns.
X.fillna(X.mean(numeric_only=True), inplace=True)

# Convert X to numpy array for Keras compatibility, explicitly setting dtype
X_np = X.values.astype(np.float32)

print("X has been converted to a purely numerical numpy array (float32) for Keras compatibility.")
print(f"Shape of X_np: {X_np.shape}, Dtype: {X_np.dtype}")

print("Starting 5-fold cross-validation...")

for fold, (train_index, val_index) in enumerate(kf.split(X_np, y_encoded)):
    print(f"\n--- Fold {fold+1}/5 ---")

    # X_train and X_val are already slices of X_np (which is float32), so no further .astype() is needed here.
    X_train, X_val = X_np[train_index], X_np[val_index]
    y_train, y_val = y_encoded[train_index], y_encoded[val_index]

    model = create_model(input_dim, num_classes)

    # Train the model
    history = model.fit(
        X_train,
        y_train,
        epochs=10,  # You can adjust the number of epochs
        batch_size=32, # You can adjust the batch size
        validation_data=(X_val, y_val),
        verbose=0 # Set to 1 or 2 for more detailed output during training
    )

    # Evaluate the model on the validation set
    loss, accuracy = model.evaluate(X_val, y_val, verbose=0)
    print(f"Fold {fold+1} - Validation Loss: {loss:.4f}, Validation Accuracy: {accuracy:.4f}")

    all_losses.append(loss)
    all_accuracies.append(accuracy)

print("\n--- Cross-validation complete ---")
print(f"Average Validation Accuracy: {np.mean(all_accuracies):.4f} (+/- {np.std(all_accuracies):.4f})")
print(f"Average Validation Loss: {np.mean(all_losses):.4f} (+/- {np.std(all_losses):.4f})")

# Plotting training history for the last fold (as an example)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy - Last Fold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss - Last Fold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

print("Cross-validation results and last fold's training history plotted.")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Get predictions for the last validation fold
y_pred_prob = model.predict(X_val)
y_pred = np.argmax(y_pred_prob, axis=1)

print("\n--- Detailed Evaluation for the Last Fold ---")

# Dynamically get unique labels and their names for the current fold
unique_labels_in_fold = np.unique(y_val)
target_names_in_fold = le.inverse_transform(unique_labels_in_fold)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_val, y_pred, labels=unique_labels_in_fold, target_names=target_names_in_fold))

# Confusion Matrix
conf_matrix = confusion_matrix(y_val, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names_in_fold, yticklabels=target_names_in_fold) # Use dynamically derived names
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Last Fold (Severity)') # Updated title
plt.show()

print("Detailed classification report and confusion matrix displayed for the last validation fold.")

In [ ]:
print("Number of columns before dropping all-NaN columns:", merged_df.shape[1])

# Identify columns with 0 non-null values
columns_to_drop_all_nan = merged_df.columns[merged_df.isnull().all()].tolist()

if columns_to_drop_all_nan:
    merged_df.drop(columns=columns_to_drop_all_nan, inplace=True)
    print(f"Dropped {len(columns_to_drop_all_nan)} columns that were entirely NaN: {columns_to_drop_all_nan}")
else:
    print("No columns were entirely NaN to drop.")

print("Number of columns after dropping all-NaN columns手間:", merged_df.shape[1])

# Re-identify numerical and categorical columns after dropping all-NaN columns
# Exclude 'Severity' from categorical features as it is now the target variable
numerical_cols = merged_df.select_dtypes(include=['number']).columns.tolist()
categorical_cols = merged_df.select_dtypes(exclude=['number']).columns.tolist()

if 'Severity' in categorical_cols:
    categorical_cols.remove('Severity') # Ensure target is not processed as a feature

print(f"Number of numerical columns (after dropping all-NaN and excluding target): {len(numerical_cols)}")
print(f"Number of categorical columns (after dropping all-NaN and excluding target): {len(categorical_cols)}")

# Re-impute remaining missing values for numerical columns with the mean
for col in numerical_cols:
    if merged_df[col].isnull().any():
        merged_df[col] = merged_df[col].fillna(merged_df[col].mean())

# Re-impute remaining missing values for categorical columns with the mode or 'Unknown'
for col in categorical_cols:
    if merged_df[col].isnull().any():
        mode_val = merged_df[col].mode()
        if not mode_val.empty:
            merged_df[col] = merged_df[col].fillna(mode_val[0])
        else:
            merged_df[col] = merged_df[col].fillna('Unknown')

print("Missing values after re-imputation (expected to be 0 for features):")
print(merged_df.drop(columns=['Severity']).isnull().sum().sum()) # Check features only

# Apply one-hot encoding to all identified categorical features (excluding 'Severity')
merged_df = pd.get_dummies(merged_df, columns=categorical_cols, drop_first=True)

print("Categorical features one-hot encoded.")
print(f"Shape of DataFrame after one-hot encoding: {merged_df.shape}")

In [ ]:
from sklearn.preprocessing import StandardScaler

# Re-identify numerical columns after one-hot encoding, excluding the target variable 'Severity'
numerical_cols = merged_df.select_dtypes(include=['number']).columns.tolist()
if 'Severity' in numerical_cols:
    numerical_cols.remove('Severity') # Ensure target is not scaled

# Initialize StandardScaler
scaler = StandardScaler()

# Apply scaling to numerical features
merged_df[numerical_cols] = scaler.fit_transform(merged_df[numerical_cols])

print("Numerical features scaled using StandardScaler.")
print("First 5 rows of scaled numerical features (excluding Severity):")
print(merged_df[numerical_cols].head())

## Summary:

### Q&A
The neural network model achieved an average validation accuracy of 0.9608 (\+/- 0.0034) and an average validation loss of 0.1635 (\+/- 0.0245) across 5-fold cross-validation. For the last validation fold, the model demonstrated good performance, with high precision, recall, and F1-score for the 'SLIGHT' severity (0.97 precision, 0.98 recall, 0.98 f1-score) and 'SERIOUS' severity (0.89 precision, 0.86 recall, 0.88 f1-score). For the 'FATAL' severity, the model showed perfect precision (1.00) but lower recall (0.43), indicating it correctly identified fatal cases when it predicted them, but missed some. This suggests the model struggled more with the highly imbalanced 'FATAL' class, which is expected due to its very low representation in the dataset.

### Data Analysis Key Findings
*   The initial dataset (`merged_collision_data.csv`) contained 29,835 rows and 99 columns. After reloading, rows with missing 'Severity' were removed, reducing the dataset to 9,679 entries. The target variable 'Severity' was successfully normalized to uppercase ('FATAL', 'SERIOUS', 'SLIGHT') and then encoded into 0-indexed numerical values.
*   The distribution of the 'Severity' target variable after normalization and cleaning was:
    *   SLIGHT: 8222
    *   SERIOUS: 1436
    *   FATAL: 21
*   Feature engineering successfully extracted 'hour_of_day' and 'month' from 'date' and 'time' columns, which were then dropped.
*   A total of 51 columns that were entirely `NaN` were dropped. Remaining numerical missing values were imputed with the mean, and categorical missing values with the mode or 'Unknown'.
*   Categorical features were one-hot encoded, leading to a DataFrame with 10,801 columns. Numerical features were scaled using `StandardScaler`.
*   The neural network showed strong performance overall, particularly for the 'SLIGHT' and 'SERIOUS' categories.

### Insights or Next Steps
*   The model's performance on the highly imbalanced 'FATAL' class (low recall) could be improved using techniques like oversampling (e.g., SMOTE), undersampling, or class weighting during training. This would help the model learn more effectively from the minority class.
*   While the model performed well on the overall accuracy, further analysis of the misclassified instances, especially for 'FATAL' cases, could provide insights into specific features that contribute to misprediction.
*   Consider performing data cleaning and standardization of the 'Severity' categories (e.g., merging 'SLIGHT' and 'Slight', 'SERIOUS' and 'Serious', 'FATAL' and 'Fatal') to reduce the number of target classes, which might simplify the classification problem and provide a clearer interpretation of the model's performance.
*   Evaluate the model on an entirely unseen test set to ensure its generalization capabilities are robust and not solely optimized for the cross-validation folds.


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

kf = KFold(n_splits=5, shuffle=True, random_state=42)

all_accuracies = []
all_losses = []

# Ensure all columns in X are numeric and handle any remaining non-numeric values
for col in X.columns:
    # Attempt to convert column to numeric, coercing any unconvertible values to NaN
    X[col] = pd.to_numeric(X[col], errors='coerce')

# After coercing, impute any newly introduced NaNs (from conversion failures)
# with the mean of their respective columns.
# We ensure to only calculate mean for numeric columns.
X.fillna(X.mean(numeric_only=True), inplace=True)

# Convert X to numpy array for Keras compatibility, explicitly setting dtype
X_np = X.values.astype(np.float32)

print("X has been converted to a purely numerical numpy array (float32) for Keras compatibility.")
print(f"Shape of X_np: {X_np.shape}, Dtype: {X_np.dtype}")

print("Starting 5-fold cross-validation...")

for fold, (train_index, val_index) in enumerate(kf.split(X_np, y_encoded)):
    print(f"\n--- Fold {fold+1}/5 ---")

    # X_train and X_val are already slices of X_np (which is float32), so no further .astype() is needed here.
    X_train, X_val = X_np[train_index], X_np[val_index]
    y_train, y_val = y_encoded[train_index], y_encoded[val_index]

    model = create_model(input_dim, num_classes)

    # Train the model
    history = model.fit(
        X_train,
        y_train,
        epochs=10,  # You can adjust the number of epochs
        batch_size=32, # You can adjust the batch size
        validation_data=(X_val, y_val),
        verbose=0 # Set to 1 or 2 for more detailed output during training
    )

    # Evaluate the model on the validation set
    loss, accuracy = model.evaluate(X_val, y_val, verbose=0)
    print(f"Fold {fold+1} - Validation Loss: {loss:.4f}, Validation Accuracy: {accuracy:.4f}")

    all_losses.append(loss)
    all_accuracies.append(accuracy)

print("\n--- Cross-validation complete ---")
print(f"Average Validation Accuracy: {np.mean(all_accuracies):.4f} (+/- {np.std(all_accuracies):.4f})")
print(f"Average Validation Loss: {np.mean(all_losses):.4f} (+/- {np.std(all_losses):.4f})")

# Plotting training history for the last fold (as an example)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy - Last Fold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss - Last Fold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

print("Cross-validation results and last fold's training history plotted.")

**Reasoning**:
The neural network achieved extremely high validation accuracy (0.9999) across all cross-validation folds. While seemingly positive, such high performance on a real-world dataset, especially with imbalanced classes, is often indicative of potential issues like data leakage or an oversimplified problem. To investigate this further and ensure the model's reliability, the next step is to evaluate the model using more detailed classification metrics (precision, recall, F1-score) and visualize the confusion matrix for the last validation fold. This will provide a deeper understanding of how the model performs across different classes and identify any potential biases or leakage.



In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Get predictions for the last validation fold
y_pred_prob = model.predict(X_val)
y_pred = np.argmax(y_pred_prob, axis=1)

print("\n--- Detailed Evaluation for the Last Fold ---")

# Dynamically get unique labels and their names for the current fold
unique_labels_in_fold = np.unique(y_val)
target_names_in_fold = le.inverse_transform(unique_labels_in_fold)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_val, y_pred, labels=unique_labels_in_fold, target_names=target_names_in_fold))

# Confusion Matrix
conf_matrix = confusion_matrix(y_val, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names_in_fold, yticklabels=target_names_in_fold) # Use dynamically derived names
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Last Fold (Severity)') # Updated title
plt.show()

print("Detailed classification report and confusion matrix displayed for the last validation fold.")

## Final Task

### Subtask:
Provide a summary of the neural network model's performance based on the 5-fold cross-validation results.


## Summary:

### Q&A
The neural network model achieved an average validation accuracy of 0.9999 (\+/- 0.0001) and an average validation loss of 0.0003 (\+/- 0.0004) across 5-fold cross-validation. For the last validation fold, the model demonstrated perfect precision, recall, and F1-score for all three `collision_severity` classes (1, 2, and 3), indicating flawless classification on that specific fold.

### Data Analysis Key Findings
*   The initial dataset (`merged_collision_data.csv`) contained 29,835 rows and 99 columns, with `collision_severity` having 4,888 missing values and a highly imbalanced distribution (20,340 for 3.0, 4,559 for 2.0, 48 for 1.0).
*   Rows with missing `collision_severity` were removed, reducing the dataset to 24,947 entries, and the target variable was converted to an integer type.
*   Feature engineering successfully extracted 'hour\_of\_day' and 'month' from 'date' and 'time' columns.
*   A total of 35 columns that were entirely `NaN` were dropped, and remaining missing values were imputed (numerical with mean, categorical with mode or 'Unknown').
*   Categorical features were one-hot encoded, expanding the feature set significantly from 64 to 7,881 columns, and numerical features were scaled using `StandardScaler`.
*   The neural network, after addressing a `ValueError` by ensuring all features in X were numeric and converting to `float32`, showed exceptional performance with an average validation accuracy of 0.9999.

### Insights or Next Steps
*   The exceptionally high model performance (near-perfect accuracy and F1-scores across all classes) warrants further investigation to rule out potential issues like data leakage, overfitting, or an overly simplified representation of the problem, which can occur with such high performance in complex, real-world data.
*   Consider evaluating the model on an entirely unseen test set to ensure its generalization capabilities are robust and not solely optimized for the cross-validation folds.


# Task
Build and evaluate a neural network model to predict collision severity using the data from `merged_collision_data.csv`, performing data preprocessing steps including handling missing values, feature engineering (extracting 'hour_of_day' and 'month'), one-hot encoding categorical features, scaling numerical features, and then performing 5-fold cross-validation to assess the model's performance with appropriate classification metrics and visualizations.

## Load Merged Data

### Subtask:
Load the `merged_collision_data.csv` into a pandas DataFrame.


**Reasoning**:
Load the `merged_collision_data.csv` file into a pandas DataFrame, display its first 5 rows, and print its shape as instructed. This will allow for initial inspection of the data.



In [ ]:
merged_df = pd.read_csv('merged_collision_data.csv', low_memory=False)
print("First 5 rows of merged_df:")
print(merged_df.head())
print(f"Shape of merged_df: {merged_df.shape}")

## Final Task

### Subtask:
Provide a summary of the neural network model's performance based on the 5-fold cross-validation results.


## Summary:

### Data Analysis Key Findings
*   The `merged_collision_data.csv` file was successfully loaded into a pandas DataFrame named `merged_df`.
*   The DataFrame consists of 29,835 rows and 99 columns.
*   An initial inspection of the first few rows revealed the presence of `NaN` values in several columns, including `did_police`, `trunk_road`, `lsoa_of_ac`, `enhanced_s`, `collisio_4`, `collisio_5`, `collisio_6`, `Pedestrian_2`, `Pedestri_1_2`, and `Casualty M`.

### Insights or Next Steps
*   The presence of missing values in multiple columns indicates that a crucial next step will be to implement strategies for handling these missing data points, such as imputation or removal, as part of the data preprocessing phase.
